In [2]:
import numpy as np

X = np.load("artifacts/X.npy")
y = np.load("artifacts/y.npy")
ft = np.load("artifacts/fruit_type.npy", allow_pickle=True)

print("Shape:", X.shape)

Shape: (13232, 1310)


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, ft_train, ft_test = train_test_split(
    X, y, ft,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (10585, 1310)
Test : (2647, 1310)


In [5]:
### scalling
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

joblib.dump(scaler, "models/scaler.pkl")

['models/scaler.pkl']

In [6]:
from sklearn.feature_selection import VarianceThreshold

var_sel = VarianceThreshold(0)

X_train = var_sel.fit_transform(X_train)
X_test = var_sel.transform(X_test)

joblib.dump(var_sel, "models/var.pkl")

print("After variance:", X_train.shape)

After variance: (10585, 1310)


In [ ]:
from sklearn.svm import SVC
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score

feature_counts = [80, 120, 160, 200]

results = []

for n in feature_counts:
    rfe = RFE(SVC(kernel='linear'), n_features_to_select=n, step=50)

    X_tr = rfe.fit_transform(X_train, y_train)
    X_te = rfe.transform(X_test)

    clf = SVC(kernel='rbf')
    clf.fit(X_tr, y_train)

    pred = clf.predict(X_te)
    acc = accuracy_score(y_test, pred)

    results.append((n, acc))
    print(f"Features: {n}, Accuracy: {acc:.4f}")

best_n = sorted(results, key=lambda x: (-x[1], x[0]))[0][0]

print("\nSelected features:", best_n)

Features: 80, Accuracy: 0.9819
Features: 120, Accuracy: 0.9849
Features: 160, Accuracy: 0.9849
Features: 200, Accuracy: 0.9841

Selected features: 120


In [9]:
rfe = RFE(SVC(kernel='linear'), n_features_to_select=best_n, step=50)

X_train = rfe.fit_transform(X_train, y_train)
X_test = rfe.transform(X_test)

import joblib
joblib.dump(rfe, "models/rfe.pkl")

print("Final shape:", X_train.shape)

Final shape: (10585, 120)


In [10]:
np.save("artifacts/X_train.npy", X_train)
np.save("artifacts/X_test.npy", X_test)
np.save("artifacts/y_train.npy", y_train)
np.save("artifacts/y_test.npy", y_test)
np.save("artifacts/ft_train.npy", ft_train)
np.save("artifacts/ft_test.npy", ft_test)

print("Saved processed data")

Saved processed data
